# Road Safety LLM for Michelin

## Datasets and Cleaning

### Crash data_LA_county

In [2]:
# import necessary libraries

import pandas as pd
import numpy as np
import ollama
from langchain.llms import Ollama
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from geopy.geocoders import Nominatim

In [9]:
# Save dataset as pandas dataframe
crash_df = pd.read_csv("/Users/ramsharauf/Desktop/BTTAI_final/Road-Safety-LLM/Datasets/Crash data_LA_county.csv")

In [10]:
# View the data
crash_df.head()

,LATITUDE,LONGITUDE,ARC_ID,YEAR,VH,CYC,PED,FATAL,SERIOUS_INJURY,MINOR_INJURY,NO_INJURY
0,34.022659,-118.438652,3815613985340226038495917846_38156133053402269...,2018.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,33.771053,-118.240303,3817593925337726257993907375_38175966053377030...,2018.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,34.025581,-118.335114,38166473753402558910272263152_3816648955340255...,2018.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,34.164879,-118.410530,3815895015341648791916620220_38158843353416488...,2019.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,33.969830,-118.347229,3816528095339698383331497996_38165261453396974...,2019.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [11]:
# Check columns names and types
print(crash_df.dtypes)

# Check number of rows and columns
print(len(crash_df))
print(len(crash_df.columns))

LATITUDE          float64
LONGITUDE         float64
ARC_ID             object
YEAR              float64
VH                float64
CYC               float64
PED               float64
FATAL             float64
SERIOUS_INJURY    float64
MINOR_INJURY      float64
NO_INJURY         float64
dtype: object
290887
11


In [12]:
# Check for any missing values
print(crash_df.isnull().sum())

LATITUDE          0
LONGITUDE         0
ARC_ID            0
YEAR              1
VH                1
CYC               1
PED               1
FATAL             1
SERIOUS_INJURY    1
MINOR_INJURY      1
NO_INJURY         1
dtype: int64


In [13]:
# Find where the missing values are located
null_row = crash_df[crash_df.isnull().any(axis=1)]
print(null_row)

         LATITUDE   LONGITUDE  \
290886  34.228191 -118.599525   

                                                  ARC_ID  YEAR  VH  CYC  PED  \
290886  381399749534228207122885743_38140053653422820612   NaN NaN  NaN  NaN   

        FATAL  SERIOUS_INJURY  MINOR_INJURY  NO_INJURY  
290886    NaN             NaN           NaN        NaN  


In [14]:
# There is only one row with most of its values missing. It would be best to just remove it
crash_df.drop(null_row.index, axis = 0, inplace = True)

In [15]:
# Check again for missing values
print(crash_df.isnull().sum())

LATITUDE          0
LONGITUDE         0
ARC_ID            0
YEAR              0
VH                0
CYC               0
PED               0
FATAL             0
SERIOUS_INJURY    0
MINOR_INJURY      0
NO_INJURY         0
dtype: int64


### Harsh Braking_Severity_Ranking_Clustering_LA_COUNTY_H10

#### Columns and Description

CLUSTER_ID:
This is a unique identifier for each cluster. A cluster represents a spatial region or area where multiple events (such as harsh accelerations) have occurred.

VEHICLE_COUNT:
The number of vehicles involved in the harsh acceleration events within each cluster. The higher the count, the more frequent the events in that area.

CENTROID:
The geographic center (lat/lon coordinates) of the cluster. This is calculated based on the geometry of the cluster and gives a representative location of the entire cluster area.

POLYGON:
A set of geographic coordinates representing the boundaries of the cluster. Each cluster is enclosed within a polygon, providing the spatial extent of where the harsh accelerations were recorded.

RANK:
The severity ranking of the cluster. This orders the clusters based on the vehicle count. Lower ranks indicate a higher vehicle count.

MULTILINESTRING_CLUSTERS:
This contains multiple lines or paths in geographic coordinates, possibly representing the driving routes or roads where harsh accelerations occurred. It shows the trajectory or multiple connected line segments within the cluster.

LENGTH_CLUSTER:
The total length (in some unit like meters) of all roads or paths involved in the cluster. This can give an idea of the physical size or extent of the cluster.

LIST_ROAD_NAME:
The names of the roads where the harsh acceleration events occurred. This helps in identifying the specific roads within the cluster and can be useful for understanding road-related driving behavior.

LIST_CITY_NAME:
The cities or metropolitan areas associated with each cluster. Since it's from Los Angeles County, most clusters might be within the Greater Los Angeles area. Beverly Hills, Carson, el segundo, Gardena, Inglewood, Los Angeles, Santa Monica, Torrance, Vernon, West Hollywood

LIST_ROAD_PRIORITY:
One or more numerical values that indicate the importance or type of the road (e.g., highways, arterial roads). 2,3,5,6,7,8,9,10,11,12,13,14,15. Still not sure what it specifically targets

H3_ID:
This is the H3 index or ID representing the hexagonal cell in the H3 framework. H3 is a spatial indexing system used to partition the globe into hexagonal cells, enabling efficient geospatial operations.

H3_GEOM:
The geometric representation (usually in the form of a polygon) of the H3 cell associated with each cluster. This defines the spatial boundaries of the hexagonal grid cell covering that cluster.

In [17]:
# Save dataset as pandas dataframe
acceleration_df = pd.read_csv("/Users/ramsharauf/Desktop/BTTAI_final/Road-Safety-LLM/Datasets/Harsh Acceleration_Severity_Ranking_Clustering_LA_COUNTY_H10.csv")

In [18]:
# Check number of rows and columns
print(len(acceleration_df))
print(len(acceleration_df.columns))

12868
12


In [19]:
acceleration_df.head()

,CLUSTER_ID,VEHICLE_COUNT,CENTROID,POLYGON,RANK,MULTILINESTRING_CLUSTERS,LENGTH_CLUSTER,LIST_ROAD_NAME,LIST_CITY_NAME,LIST_ROAD_PRIORITY,H3_ID,H3_GEOM
0,294,496,POINT(-118.267279584 34.029165823),"POLYGON((-118.26741 34.028915,-118.26751 34.02...",216,"MULTILINESTRING((-118.267107 34.029412,-118.26...",156.919375,"[\n ""S Broadway""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 9\n],"""8a29a1d6611ffff""","POLYGON((-118.267653416 34.028666453,-118.2670..."
1,2604,445,POINT(-118.34053631 34.031118907),"POLYGON((-118.34054 34.030975,-118.34057 34.03...",3516,"MULTILINESTRING((-118.340532 34.031128,-118.34...",192.446380,"[\n ""Buckingham Road"",\n ""Buckingham Rd""\n]","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 7\n],"""8a29a1992987fff""","POLYGON((-118.340945887 34.029933312,-118.3402..."
2,2604,445,POINT(-118.34053631 34.031118907),"POLYGON((-118.34054 34.030975,-118.34057 34.03...",3516,"MULTILINESTRING((-118.340532 34.031128,-118.34...",192.446380,"[\n ""Buckingham Road"",\n ""Buckingham Rd""\n]","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 7\n],"""8a29a199299ffff""","POLYGON((-118.340474043 34.031133348,-118.3398..."
3,1899,144,POINT(-118.557961122 34.239184329),"POLYGON((-118.558014 34.23899,-118.55812 34.23...",5563,"MULTILINESTRING((-118.55795 34.2409,-118.55795...",600.081703,"[\n ""Prairie St"",\n ""Shirley Ave""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 7\n],"""8a29a18f36a7fff""","POLYGON((-118.55710626 34.237923709,-118.55645..."
4,1899,144,POINT(-118.557961122 34.239184329),"POLYGON((-118.558014 34.23899,-118.55812 34.23...",5563,"MULTILINESTRING((-118.55795 34.2409,-118.55795...",600.081703,"[\n ""Prairie St"",\n ""Shirley Ave""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 7\n],"""8a29a18f36affff""","POLYGON((-118.558116407 34.238831849,-118.5574..."


In [20]:
# Check columns names and types
print(acceleration_df.dtypes)

CLUSTER_ID                    int64
VEHICLE_COUNT                 int64
CENTROID                     object
POLYGON                      object
RANK                          int64
MULTILINESTRING_CLUSTERS     object
LENGTH_CLUSTER              float64
LIST_ROAD_NAME               object
LIST_CITY_NAME               object
LIST_ROAD_PRIORITY           object
H3_ID                        object
H3_GEOM                      object
dtype: object


In [21]:
# Get all the columns with type object and check their values
acc_obj_cols = acceleration_df.columns[acceleration_df.dtypes == object]
print(acc_obj_cols)


Index(['CENTROID', 'POLYGON', 'MULTILINESTRING_CLUSTERS', 'LIST_ROAD_NAME',
       'LIST_CITY_NAME', 'LIST_ROAD_PRIORITY', 'H3_ID', 'H3_GEOM'],
      dtype='object')


In [22]:
# See what they look like in the dataframe
acceleration_df[acc_obj_cols]

# Might possibly need to be converted/cleaned depending on model performance

,CENTROID,POLYGON,MULTILINESTRING_CLUSTERS,LIST_ROAD_NAME,LIST_CITY_NAME,LIST_ROAD_PRIORITY,H3_ID,H3_GEOM
0,POINT(-118.267279584 34.029165823),"POLYGON((-118.26741 34.028915,-118.26751 34.02...","MULTILINESTRING((-118.267107 34.029412,-118.26...","[\n ""S Broadway""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 9\n],"""8a29a1d6611ffff""","POLYGON((-118.267653416 34.028666453,-118.2670..."
1,POINT(-118.34053631 34.031118907),"POLYGON((-118.34054 34.030975,-118.34057 34.03...","MULTILINESTRING((-118.340532 34.031128,-118.34...","[\n ""Buckingham Road"",\n ""Buckingham Rd""\n]","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 7\n],"""8a29a1992987fff""","POLYGON((-118.340945887 34.029933312,-118.3402..."
2,POINT(-118.34053631 34.031118907),"POLYGON((-118.34054 34.030975,-118.34057 34.03...","MULTILINESTRING((-118.340532 34.031128,-118.34...","[\n ""Buckingham Road"",\n ""Buckingham Rd""\n]","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 7\n],"""8a29a199299ffff""","POLYGON((-118.340474043 34.031133348,-118.3398..."
3,POINT(-118.557961122 34.239184329),"POLYGON((-118.558014 34.23899,-118.55812 34.23...","MULTILINESTRING((-118.55795 34.2409,-118.55795...","[\n ""Prairie St"",\n ""Shirley Ave""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 7\n],"""8a29a18f36a7fff""","POLYGON((-118.55710626 34.237923709,-118.55645..."
4,POINT(-118.557961122 34.239184329),"POLYGON((-118.558014 34.23899,-118.55812 34.23...","MULTILINESTRING((-118.55795 34.2409,-118.55795...","[\n ""Prairie St"",\n ""Shirley Ave""\n]","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 7\n],"""8a29a18f36affff""","POLYGON((-118.558116407 34.238831849,-118.5574..."
...,...,...,...,...,...,...,...,...
12863,POINT(-118.215502304 34.069039345),"MULTIPOLYGON(((-118.21562 34.068665,-118.21559...","MULTILINESTRING((-118.215521 34.068704,-118.21...","[\n ""Daly St"",\n ""Daly Street""\n]","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 11\n],"""8a29a1d73d2ffff""","POLYGON((-118.214901075 34.068810344,-118.2142..."
12864,POINT(-118.30898763 33.956258725),"POLYGON((-118.30898 33.95622,-118.30901 33.956...",NaN,"[\n ""South Western Avenue"",\n ""S Western Ave...","[\n ""Los Angeles"",\n ""Los Angeles--Long Beac...",[\n 11\n],"""8a29a56ccce7fff""","POLYGON((-118.30883718 33.955267853,-118.30818..."
12865,POINT(-118.590524868 34.274387108),"POLYGON((-118.59071 34.274345,-118.59073 34.27...","MULTILINESTRING((-118.590436786 34.274440579,-...","[\n ""74680e53-7845-4c70-893a-19f7901bbc64"",\n...","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 14\n],"""8a29a18c264ffff""","POLYGON((-118.589579308 34.273586203,-118.5889..."
12866,POINT(-118.590524868 34.274387108),"POLYGON((-118.59071 34.274345,-118.59073 34.27...","MULTILINESTRING((-118.590436786 34.274440579,-...","[\n ""74680e53-7845-4c70-893a-19f7901bbc64"",\n...","[\n ""Los Angeles--Long Beach--Anaheim, CA""\n]",[\n 14\n],"""8a29a18dc937fff""","POLYGON((-118.5910595 34.273294856,-118.590409..."


In [23]:
# Check for any missing values
print(acceleration_df.isnull().sum())

CLUSTER_ID                   0
VEHICLE_COUNT                0
CENTROID                     0
POLYGON                      0
RANK                         0
MULTILINESTRING_CLUSTERS    44
LENGTH_CLUSTER              44
LIST_ROAD_NAME               0
LIST_CITY_NAME               0
LIST_ROAD_PRIORITY           0
H3_ID                        0
H3_GEOM                      0
dtype: int64


In [24]:
# Find where the missing values are located
null_row_acc = acceleration_df[acceleration_df.isnull().any(axis=1)]
print(null_row_acc)

       CLUSTER_ID  VEHICLE_COUNT                            CENTROID  \
448          5561            272  POINT(-118.383382553 34.172181777)   
449          5561            272  POINT(-118.383382553 34.172181777)   
1064         4469            454  POINT(-118.300300218 33.970926422)   
1541         2621            705  POINT(-118.446942894 34.224772918)   
1961         1446           2231  POINT(-118.396138837 33.957064084)   
2253         3857            286   POINT(-118.396668224 34.15763189)   
2254         3857            286   POINT(-118.396668224 34.15763189)   
2255         3857            286   POINT(-118.396668224 34.15763189)   
2269         5844            143  POINT(-118.273881813 33.949170804)   
2270         5844            143  POINT(-118.273881813 33.949170804)   
2466         5829             41  POINT(-118.588751494 34.242746552)   
2659         4342            109  POINT(-118.370247574 34.172189083)   
3168         1725            273  POINT(-118.226385789 34.142783

In [25]:
# Get the number of distinct values for each column
distinct_counts = acceleration_df.nunique()

# Print the number of distinct values for each column
for column, count in distinct_counts.items():
    print(f"Number of distinct values in column '{column}': {count}")

Number of distinct values in column 'CLUSTER_ID': 6345
Number of distinct values in column 'VEHICLE_COUNT': 1337
Number of distinct values in column 'CENTROID': 6345
Number of distinct values in column 'POLYGON': 6345
Number of distinct values in column 'RANK': 6345
Number of distinct values in column 'MULTILINESTRING_CLUSTERS': 6312
Number of distinct values in column 'LENGTH_CLUSTER': 6312
Number of distinct values in column 'LIST_ROAD_NAME': 3782
Number of distinct values in column 'LIST_CITY_NAME': 42
Number of distinct values in column 'LIST_ROAD_PRIORITY': 40
Number of distinct values in column 'H3_ID': 10127
Number of distinct values in column 'H3_GEOM': 10127


## Train Model

In [27]:
# Model found for geospatial data
geospatial_llm  = Ollama(model = 'ALIENTELLIGENCE/surveyingandmapping', callback_manager= CallbackManager([StreamingStdOutCallbackHandler()]))

accident_map_prompt = PromptTemplate(input_variables = ["dataframe"], template= "Using this dataset, {dataframe}, give a python list of the addresses of the areas with the most pedestrian accidents.")

chain = LLMChain(llm = geospatial_llm, prompt = accident_map_prompt)

print(chain.run({"dataframe": crash_df}))



/var/folders/j3/b6phw13967n5dh5kr6q_vf_40000gn/T/ipykernel_3245/3062890150.py:2: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  geospatial_llm  = Ollama(model = 'ALIENTELLIGENCE/surveyingandmapping', callback_manager= CallbackManager([StreamingStdOutCallbackHandler()]))


To get the top N areas with the most pedestrian accidents, we need to group the data by area (ARC_ID) and sum up the number of PED (pedestrian accidents). Then, we can sort the data by this sum in descending order and select the top N rows.

Here is a Python function using

KeyboardInterrupt: 

In [46]:
crash_df.sort_values(by = 'LATITUDE')

,LATITUDE,LONGITUDE,ARC_ID,YEAR,VH,CYC,PED,FATAL,SERIOUS_INJURY,MINOR_INJURY,NO_INJURY
70129,33.327965,-118.339500,3816604975333280181124323221_38166039253332783...,2022.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
244188,33.327965,-118.339500,381660392533327839122864432_381660497533328018...,2022.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
93816,33.333515,-118.334831,3816654975333336324359576503_38166512553333350...,2023.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
195536,33.334198,-118.312553,381687501533334227122983131_381687356533334172...,2020.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
21731,33.334198,-118.312553,3816873565333341721852175375_38168750153333422...,2020.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
204529,34.820675,-118.156082,3818450225348137701903131273_38184393853482067...,2024.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
180376,34.820675,-118.156082,38184393853482067589368043_3818450225348137701...,2023.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
186107,34.820675,-118.156082,38184393853482067589368043_3818450225348137701...,2023.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
180375,34.820675,-118.156082,3818450225348137701903131273_38184393853482067...,2023.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

# Load the dataset (assuming crash_df is predefined)
df = crash_df

# Calculate the sum of all accident types for each unique combination of LATITUDE and LONGITUDE
accidents_by_location = df.groupby(['LATITUDE', 'LONGITUDE']).sum()

# Filter out areas with more than 3 pedestrian accidents (of any severity)
locations_with_ped_accidents = accidents_by_location[accidents_by_location['PED'] > 3]

# Sort by the 'PED' column in descendsing order and take the top 10
top_10_locations = locations_with_ped_accidents.sort_values(by='PED', ascending=False).head(10)

# Format the addresses into a list
addresses = top_10_locations.index.tolist()
print(addresses)

if not addresses:
    print('Error: No locations found with more than 3 pedestrian accidents')
    raise ValueError('No matching locations')

# Initialize geolocator
geoloc = Nominatim(user_agent='GetLoc')

# List to store the addresses
add_list = []

# Get the address for each location
for address in addresses:
    try:
        # Ensure that the address is a tuple of (latitude, longitude)
        lat, lon = address  # address is a tuple of (latitude, longitude)
        
        # Pass the latitude and longitude as a tuple
        locname = geoloc.reverse((lat, lon), language='en', timeout=10)
        
        # If geoloc returns a result, append it to the list
        if locname:
            add_list.append(locname.address)
        else:
            add_list.append('Address not found')
    except Exception as e:
        add_list.append(f'Error: {e}')
    
    # Adding a delay to avoid hitting API rate limits
    time.sleep(1)

# Print the results
for location, address in zip(addresses, add_list):
    print(f"Location: {location} => Address: {address}")




[(33.974658966, -118.278327942), (34.003871918, -118.265266418), (34.193851471, -118.53604126), (33.775352478, -118.167671204), (34.021530151, -118.355613708), (34.235530853, -118.467712402), (34.035049438, -118.238639832), (34.051738739, -118.279830933), (33.860221863, -118.151176453), (33.974681854, -118.27394104)]
Location: (33.974658966, -118.278327942) => Address: West Florence Avenue, Florence, Los Angeles, Los Angeles County, California, 90003, United States
Location: (34.003871918, -118.265266418) => Address: El Pollo Loco, 4405, Avalon Boulevard, South Park, Los Angeles, Los Angeles County, California, 90011, United States
Location: (34.193851471, -118.53604126) => Address: Vanowen Street, Reseda, Los Angeles, Los Angeles County, California, 91335, United States
Location: (33.775352478, -118.167671204) => Address: East 7th Street, Long Beach, Los Angeles County, California, 90804, United States
Location: (34.021530151, -118.355613708) => Address: Obama Boulevard, Village Green